In [34]:
import sqlite3
import pandas as pd

In [15]:
# TODO: user input for group name
GNAME = "Grupo Deus é Amor" # TODO

conn = sqlite3.connect('dataset/railway.sqlite')
cursor = conn.cursor()

# Get all tags for a group
query_tags = "SELECT name FROM tags WHERE group_id = (SELECT id FROM groups WHERE name = ?)"
cursor.execute(query_tags, (GNAME,))
tags = cursor.fetchall()
tags = [tag[0] for tag in tags]

# TODO: drop down menu for tags

# TODO: user input for tag selection(s)
TAG = ['quinta', 'quaresma'] # TODO


In [30]:
def get_placeholder(target_list):
    return ','.join(['?'] * len(target_list))

In [35]:
# Get all event ids for the group with the specified tags
placeholder_tags = get_placeholder(TAG)
query_eid = f"\
    SELECT DISTINCT e.id \
    FROM events AS e \
    JOIN event_tags AS et ON et.event_id = e.id \
    JOIN tags ON tags.id = et.tag_id \
    WHERE e.group_id = (SELECT id FROM groups WHERE name = ?)\
        AND tags.name IN ({placeholder_tags})"
    
cursor.execute(query_eid, [GNAME]+TAG)
event_ids = cursor.fetchall()
event_ids = [eid[0] for eid in event_ids]

# Get participant & checkin info for the events
placeholder_eids = get_placeholder(event_ids)
query = f"\
    SELECT e.name AS event_name, e.start_date_time AS event_time, \
        p.id AS participant_id, p.full_name AS participant_name, c.timestamp AS checkin_time, \
        p.birth_date AS participant_birth, p.gender AS participant_gender \
    FROM events AS e \
    JOIN check_ins AS c ON c.event_id = e.id \
    JOIN participants AS p ON p.id = c.participant_id \
    WHERE e.id IN ({placeholder_eids})" 
# cursor.execute(query, event_ids)
# results = cursor.fetchall()
df = pd.read_sql_query(query, conn, params=event_ids)

In [37]:
conn.close()